In [2]:
#Imports and NLTK Setup

In [2]:
import pandas as pd
import re
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Setup NLTK
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abdul\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
#Loading the Dataset

In [3]:
# Sentiment140 column structure
cols = ['target', 'ids', 'date', 'flag', 'user', 'text']

print("Loading dataset...")
df = pd.read_csv('training.1600000.processed.noemoticon.csv', 
                 encoding='ISO-8859-1', 
                 names=cols)
df = df.drop_duplicates()
df = df.drop(columns=['flag'])


# Display to verify
print(df.head())

Loading dataset...
   target         ids                          date             user  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  _TheSpecialOne_   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009    scotthamilton   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009         mattycus   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009          ElleCTF   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009           Karoli   

                                                text  
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1  is upset that he can't update his Facebook by ...  
2  @Kenichan I dived many times for the ball. Man...  
3    my whole body feels itchy and like its on fire   
4  @nationwideclass no, it's not behaving at all....  


In [ ]:
Mapping the Labels

In [4]:
# Map 0 -> 0 and 4 -> 1
df['sentiment_label'] = df['target'].map({0: 0, 4: 1})

# CRITICAL CHECK: Make sure you have two classes now
print("Class distribution:")
print(df['sentiment_label'].value_counts())

Class distribution:
sentiment_label
0    800000
1    800000
Name: count, dtype: int64


In [ ]:
#Cleaning the Data

In [5]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # 1. Your HTML removal
    text = re.sub(r'<.*?>', ' ', text)
    # 2. Your non-alphabetic removal
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # 3. Your lowercase conversion
    text = text.lower()
    # 4. Your stemming and stopword logic
    words = [stemmer.stem(word) for word in text.split() if word not in stop_words]
    return ' '.join(words)

# Optional: Uncomment the line below to test on 100k rows first to save time
# df = df.sample(100000, random_state=42)

print("Cleaning text... (This takes a few minutes for 1.6M rows)")
df['clean'] = df['text'].apply(clean_text)
print("Cleaning complete.")

Cleaning text... (This takes a few minutes for 1.6M rows)
Cleaning complete.


In [ ]:
#Splitting Data

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], 
    df['sentiment_label'], 
    test_size=0.2, 
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 1280000
Testing samples: 320000


In [56]:
#TF-IDF Vectorization

In [7]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)
print("Vectorization complete.")

Vectorization complete.


In [ ]:
#Training Model

In [ ]:
lr_model = LogisticRegression(C=1.0, max_iter=500, solver='liblinear')

print("Training model...")
lr_model.fit(X_train_vec, y_train)
print("Training complete.")

Training model...


In [ ]:
#Evaluation and Saving

In [1]:
preds = lr_model.predict(X_test_vec)

print(f"Accuracy: {accuracy_score(y_test, preds):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, preds))

# Save models
joblib.dump(lr_model, 'sentiment_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

print("Models saved successfully!")

NameError: name 'lr_model' is not defined